# Information Retrieval Dataset Builder

This notebook builds a dataset of Arxiv papers using the SemanticScholar and Arxiv APIs. It crawls papers by following references, prioritizing highly-cited papers from 2020 onwards.

In [ ]:
from google.colab import drive
import os
import json
import time
import requests
from datetime import datetime
from collections import Counter
import xml.etree.ElementTree as ET

# Mount Google Drive
drive.mount('/content/drive')

# Configuration
DATASET_DIR = '/content/drive/MyDrive/information_retrieval/'
DATASET_PATH = os.path.join(DATASET_DIR, 'arxiv_dataset.json')
INITIAL_ARXIV_ID = '2303.08774' # GPT-4 Technical Report

if not os.path.exists(DATASET_DIR):
    os.makedirs(DATASET_DIR)
    print(f"Created directory: {DATASET_DIR}")

## API Clients

In [ ]:
class SemanticScholarClient:
    BASE_URL = "https://api.semanticscholar.org/graph/v1/paper/"
    
    def __init__(self, api_key=None):
        self.headers = {"x-api-key": api_key} if api_key else {}
        self.last_request_time = 0
        self.rate_limit = 1.1  # Slightly above 1s to be safe

    def _wait_for_rate_limit(self):
        elapsed = time.time() - self.last_request_time
        if elapsed < self.rate_limit:
            time.sleep(self.rate_limit - elapsed)

    def get_paper_metadata(self, arxiv_id, retries=2):
        for attempt in range(retries + 1):
            self._wait_for_rate_limit()
            try:
                fields = "externalIds,title,abstract,year"
                url = f"{self.BASE_URL}ArXiv:{arxiv_id}?fields={fields}"
                response = requests.get(url, headers=self.headers, timeout=15)
                self.last_request_time = time.time()
                
                if response.status_code == 200: return response.json(), None
                if response.status_code == 429:
                    wait_time = (attempt + 1) * 5
                    print(f"[SS Metadata] 429 Rate Limit. Waiting {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                if response.status_code == 404: return None, "Not Found"
                return None, f"Error {response.status_code}"
            except Exception as e:
                if attempt < retries: continue
                return None, str(e)
        return None, "Max retries exceeded"

    def get_references(self, arxiv_id, retries=2):
        all_references = []
        offset = 0
        limit = 1000
        
        for attempt in range(retries + 1):
            self._wait_for_rate_limit()
            try:
                url = f"{self.BASE_URL}ArXiv:{arxiv_id}/references?fields=citedPaper.externalIds&limit={limit}&offset={offset}"
                response = requests.get(url, headers=self.headers, timeout=15)
                self.last_request_time = time.time()
                
                if response.status_code == 200:
                    data = response.json()
                    refs = data.get('data', [])
                    for ref in refs:
                        cited = ref.get('citedPaper', {})
                        arxiv_ref = cited.get('externalIds', {}).get('ArXiv')
                        if arxiv_ref: all_references.append(arxiv_ref)
                    return all_references, None
                
                if response.status_code == 429:
                    wait_time = (attempt + 1) * 5
                    print(f"[SS Refs] 429 Rate Limit. Waiting {wait_time}s...")
                    time.sleep(wait_time)
                    continue
                if response.status_code == 404: return [], "Not Found"
                return [], f"Error {response.status_code}"
            except Exception as e:
                if attempt < retries: continue
                return [], str(e)
        return [], "Max retries exceeded"

class ArxivClient:
    BASE_URL = "http://export.arxiv.org/api/query?id_list="
    
    def __init__(self):
        self.last_request_time = 0
        self.rate_limit = 3.0 # Arxiv is strict

    def get_paper(self, arxiv_id):
        elapsed = time.time() - self.last_request_time
        if elapsed < self.rate_limit:
            time.sleep(self.rate_limit - elapsed)
            
        try:
            url = f"{self.BASE_URL}{arxiv_id}"
            response = requests.get(url, timeout=15)
            self.last_request_time = time.time()
            
            if response.status_code != 200:
                return None, f"Error {response.status_code}"
                
            root = ET.fromstring(response.content)
            namespace = {'atom': 'http://www.w3.org/2005/Atom'}
            entry = root.find('atom:entry', namespace)
            
            if entry is None or entry.find('atom:id', namespace) is None:
                return None, "Not Found"
                
            title_elem = entry.find('atom:title', namespace)
            summary_elem = entry.find('atom:summary', namespace)
            published_elem = entry.find('atom:published', namespace)
            
            title = title_elem.text.strip().replace('\n', ' ') if title_elem is not None else ""
            abstract = summary_elem.text.strip().replace('\n', ' ') if summary_elem is not None else ""
            published = published_elem.text if published_elem is not None else "2020-01-01"
            year = int(published[:4])
            
            return {
                'title': title,
                'abstract': abstract,
                'year': year,
                'arxivID': arxiv_id
            }, None
        except Exception as e:
            return None, str(e)

## Retrieval Logic

In [ ]:
def process_paper(arxiv_id, ss_client, arxiv_client, dataset):
    """Fetches paper data and its references, updating the dataset."""
    if arxiv_id in dataset and (dataset[arxiv_id]['metadata']['success'] or dataset[arxiv_id]['metadata']['attempted']):
        return False, "Already attempted"

    print(f"---> Processing paper: {arxiv_id}")
    
    # Try Semantic Scholar first
    data, ss_error = ss_client.get_paper_metadata(arxiv_id)
    source = "SemanticScholar"
    references = []
    
    if data:
        # Get references via separate endpoint
        refs, ref_error = ss_client.get_references(arxiv_id)
        if refs: references = refs
        if ref_error: print(f"      [SS References Warning] {ref_error}")
    else:
        print(f"      [SS Metadata Failed] {ss_error}. Falling back to Arxiv API...")
        # Fallback to Arxiv API for basic metadata if SS fails
        data, arxiv_error = arxiv_client.get_paper(arxiv_id)
        source = "Arxiv"
        if not data:
            print(f"      [Arxiv Failed] {arxiv_error}. Giving up.")
            dataset[arxiv_id] = {
                'arxivID': arxiv_id,
                'metadata': {
                    'fetched_at': datetime.now().isoformat(),
                    'attempted': True,
                    'success': False,
                    'error': f"SS: {ss_error} | Arxiv: {arxiv_error}"
                }
            }
            return False, "Failed all APIs"
            
    # Check year filter (2020+)
    year = data.get('year')
    if year and year < 2020:
        print(f"      [Skipped] Published in {year} (Source: {source})")
        dataset[arxiv_id] = {
            'arxivID': arxiv_id,
            'metadata': {
                'year': year,
                'fetched_at': datetime.now().isoformat(),
                'attempted': True,
                'success': False,
                'error': f"Skipped: Published in {year}"
            }
        }
        return False, f"Skipped: {year}"

    print(f"      [Success] Found {len(references)} references via {source}.")
    if source == "Arxiv":
        print("      (Note: Arxiv API does not provide reference data)")

    # Update dataset
    dataset[arxiv_id] = {
        'arxivID': arxiv_id,
        'title': data.get('title', ''),
        'abstract': data.get('abstract', ''),
        'references': references,
        'metadata': {
            'year': year,
            'fetched_at': datetime.now().isoformat(),
            'attempted': True,
            'success': True,
            'source': source
        }
    }
    
    return True, None

In [ ]:
def get_next_batch(dataset, limit=100):
    """Identifies the next set of arxivIDs to fetch based on citation frequency."""
    referenced_ids = []
    for paper_id, data in dataset.items():
        if data.get('metadata', {}).get('success'):
            referenced_ids.extend(data.get('references', []))
            
    # Filter out IDs already in dataset (successfully or attempted)
    candidates = [rid for rid in referenced_ids if rid not in dataset]
    
    # Count citations among candidates
    counts = Counter(candidates)
    
    # Return top N most cited papers
    return [item[0] for item in counts.most_common(limit)]

## Main Execution

In [ ]:
def main():
    # Initialize clients
    ss_client = SemanticScholarClient()
    arxiv_client = ArxivClient()
    
    # Load dataset
    if os.path.exists(DATASET_PATH):
        with open(DATASET_PATH, 'r') as f:
            dataset = json.load(f)
        print(f"Loaded dataset with {len(dataset)} entries.")
    else:
        dataset = {}
        print("Created new dataset.")

    new_papers_count = 0
    error_count = 0
    ERROR_LIMIT = 5
    TARGET_NEW_PAPERS = 100
    SAVE_EVERY = 10

    print(f"=== Starting run starting with seed {INITIAL_ARXIV_ID} ===")
    
    # Step 1: Initial Arxiv ID
    if INITIAL_ARXIV_ID not in dataset:
        success, error = process_paper(INITIAL_ARXIV_ID, ss_client, arxiv_client, dataset)
        if success:
            new_papers_count += 1
        elif error and "Skipped" not in error:
            error_count += 1
            
    # Step 2: Immediate references of the seed
    if INITIAL_ARXIV_ID in dataset and dataset[INITIAL_ARXIV_ID].get('metadata', {}).get('success'):
        seed_refs = dataset[INITIAL_ARXIV_ID].get('references', [])
        if not seed_refs:
             print("!!! WARNING: Seed paper has no references. Crawler might stall unless dataset already contains other referenced papers.")
             
        for ref_id in seed_refs:
            if new_papers_count >= TARGET_NEW_PAPERS or error_count >= ERROR_LIMIT:
                break
            if ref_id not in dataset:
                success, error = process_paper(ref_id, ss_client, arxiv_client, dataset)
                if success:
                    new_papers_count += 1
                elif error and "Skipped" not in error:
                    error_count += 1
                
                if new_papers_count % SAVE_EVERY == 0:
                    with open(DATASET_PATH, 'w') as f:
                        json.dump(dataset, f, indent=2)

    # Step 3: Prioritized expansion
    while new_papers_count < TARGET_NEW_PAPERS and error_count < ERROR_LIMIT:
        next_batch = get_next_batch(dataset, limit=TARGET_NEW_PAPERS - new_papers_count)
        if not next_batch:
            print("No more candidates to fetch in this session.")
            break
            
        for arxiv_id in next_batch:
            if new_papers_count >= TARGET_NEW_PAPERS or error_count >= ERROR_LIMIT:
                break
            success, error = process_paper(arxiv_id, ss_client, arxiv_client, dataset)
            if success:
                new_papers_count += 1
            elif error and "Skipped" not in error:
                error_count += 1
            
            if new_papers_count % SAVE_EVERY == 0:
                with open(DATASET_PATH, 'w') as f:
                    json.dump(dataset, f, indent=2)

    # Save final dataset
    with open(DATASET_PATH, 'w') as f:
        json.dump(dataset, f, indent=2)
    
    print(f"=== Run complete. Added {new_papers_count} new papers. Total: {len(dataset)}. Errors: {error_count}. ===")

if __name__ == "__main__":
    main()